# EFL Championship Half-Time Draw Prediction - Model Training & Evaluation

This notebook implements **Step 3** of the README workflow: training and evaluating models for half-time draw prediction.

## Objectives
1. Load the processed dataset and perform train/validation/test split
2. Train a **Logistic Regression** baseline model
3. Train an **LSTM** sequence model  
4. Compare model performance using ROC-AUC, Brier score, and calibration
5. Analyze feature importance and model interpretability
6. Save trained models and metadata

Let's build production-ready models for half-time draw prediction!

## 1. Import Required Libraries

In [2]:
# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn for preprocessing and baseline model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    brier_score_loss, log_loss, accuracy_score,
    classification_report, confusion_matrix
)
from sklearn.calibration import calibration_curve

# PyTorch for LSTM
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

ModuleNotFoundError: No module named 'torch'

## 2. Load Processed Dataset

Load our carefully prepared dataset with all 12 features and the half-time draw target.

In [ ]:
# Load the processed dataset
df = pd.read_parquet('../data/processed/dataset.parquet')

print("📊 Dataset Loaded Successfully!")
print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Target distribution: {df['y_ht_draw'].value_counts().to_dict()}")

# Define feature columns and target
feature_cols = [
    'home_gf_r5', 'home_ga_r5', 'home_gd_r5',
    'away_gf_r5', 'away_ga_r5', 'away_gd_r5',
    'log_home_win_odds', 'log_draw_odds', 'log_away_win_odds',
    'home_days_since_last', 'away_days_since_last', 'month'
]
target_col = 'y_ht_draw'

print(f"\n📋 Features ({len(feature_cols)}): {feature_cols}")
print(f"🎯 Target: {target_col}")

# Remove rows with missing features
df_clean = df.dropna(subset=feature_cols + [target_col])
print(f"\n🧹 After removing missing values: {len(df_clean):,} matches ({len(df_clean)/len(df)*100:.1f}%)")
print(f"Draw rate: {df_clean[target_col].mean():.1%}")

## 3. Temporal Train/Validation/Test Split

Following best practices for time-series data, we split chronologically to prevent data leakage.

In [ ]:
# Sort by date to ensure chronological order
df_clean = df_clean.sort_values('Date').reset_index(drop=True)

# Split: 70% train, 15% validation, 15% test
train_size = int(0.70 * len(df_clean))
val_size = int(0.15 * len(df_clean))

train_df = df_clean.iloc[:train_size].copy()
val_df = df_clean.iloc[train_size:train_size+val_size].copy()
test_df = df_clean.iloc[train_size+val_size:].copy()

print("⏱️ TEMPORAL SPLIT (Chronological):")
print(f"\n📚 Training Set:")
print(f"  Size: {len(train_df):,} matches ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f"  Date range: {train_df['Date'].min().date()} to {train_df['Date'].max().date()}")
print(f"  Draw rate: {train_df[target_col].mean():.1%}")

print(f"\n🔍 Validation Set:")
print(f"  Size: {len(val_df):,} matches ({len(val_df)/len(df_clean)*100:.1f}%)")
print(f"  Date range: {val_df['Date'].min().date()} to {val_df['Date'].max().date()}")
print(f"  Draw rate: {val_df[target_col].mean():.1%}")

print(f"\n🧪 Test Set:")
print(f"  Size: {len(test_df):,} matches ({len(test_df)/len(df_clean)*100:.1f}%)")
print(f"  Date range: {test_df['Date'].min().date()} to {test_df['Date'].max().date()}")
print(f"  Draw rate: {test_df[target_col].mean():.1%}")

# Extract features and targets
X_train = train_df[feature_cols].values
y_train = train_df[target_col].values

X_val = val_df[feature_cols].values
y_val = val_df[target_col].values

X_test = test_df[feature_cols].values
y_test = test_df[target_col].values

print(f"\n✅ Data split complete!")
print(f"Feature matrix shapes: Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")

## 4. Feature Scaling

Standardize features for better model performance (especially important for neural networks).

In [ ]:
# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("📏 Feature Scaling Applied:")
print(f"Scaler fitted on training data only (prevents leakage)")
print(f"\nFeature means (after scaling): {X_train_scaled.mean(axis=0).round(2)}")
print(f"Feature stds (after scaling): {X_train_scaled.std(axis=0).round(2)}")
print(f"✅ All features standardized to mean=0, std=1")

## 5. Train Logistic Regression Baseline

A simple, interpretable baseline model that's fast to train and provides feature importance.

In [ ]:
print("🎯 Training Logistic Regression Baseline...")

# Train logistic regression with L2 regularization
lr_model = LogisticRegression(
    C=1.0,  # Regularization strength
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)

lr_model.fit(X_train_scaled, y_train)

# Make predictions
y_train_pred_lr = lr_model.predict_proba(X_train_scaled)[:, 1]
y_val_pred_lr = lr_model.predict_proba(X_val_scaled)[:, 1]
y_test_pred_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate on all sets
train_auc_lr = roc_auc_score(y_train, y_train_pred_lr)
val_auc_lr = roc_auc_score(y_val, y_val_pred_lr)
test_auc_lr = roc_auc_score(y_test, y_test_pred_lr)

train_brier_lr = brier_score_loss(y_train, y_train_pred_lr)
val_brier_lr = brier_score_loss(y_val, y_val_pred_lr)
test_brier_lr = brier_score_loss(y_test, y_test_pred_lr)

print(f"✅ Logistic Regression Training Complete!")
print(f"\n📊 Performance Metrics:")
print(f"  ROC-AUC  - Train: {train_auc_lr:.4f}, Val: {val_auc_lr:.4f}, Test: {test_auc_lr:.4f}")
print(f"  Brier    - Train: {train_brier_lr:.4f}, Val: {val_brier_lr:.4f}, Test: {test_brier_lr:.4f}")

# Feature importance
print(f"\n🔍 Feature Importance (Logistic Regression Coefficients):")
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr_model.coef_[0],
    'abs_coefficient': np.abs(lr_model.coef_[0])
}).sort_values('abs_coefficient', ascending=False)

for idx, row in feature_importance.head(10).iterrows():
    print(f"  {row['feature']}: {row['coefficient']:.4f}")

## 6. Define LSTM Model Architecture

Create a compact LSTM model to capture temporal patterns in the feature sequence.

In [ ]:
class HalfTimeDrawLSTM(nn.Module):
    """
    LSTM model for half-time draw prediction.
    
    Architecture:
    - Input: Sequence of features (batch_size, seq_len, num_features)
    - LSTM layer(s) to capture temporal dependencies
    - Fully connected layers with dropout
    - Sigmoid output for binary classification
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super(HalfTimeDrawLSTM, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # LSTM forward pass
        # x shape: (batch_size, seq_len, input_size)
        lstm_out, (hn, cn) = self.lstm(x)
        
        # Take the output from the last time step
        # lstm_out shape: (batch_size, seq_len, hidden_size)
        last_output = lstm_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Fully connected layers
        out = self.fc1(last_output)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        
        return out.squeeze()

# Model hyperparameters
input_size = len(feature_cols)
hidden_size = 64
num_layers = 2
dropout = 0.3
learning_rate = 0.001
batch_size = 64
num_epochs = 50

print(f"🧠 LSTM Model Architecture:")
print(f"  Input size: {input_size} features")
print(f"  Hidden size: {hidden_size}")
print(f"  Number of LSTM layers: {num_layers}")
print(f"  Dropout: {dropout}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Epochs: {num_epochs}")

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm_model = HalfTimeDrawLSTM(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout
).to(device)

print(f"\n📊 Model Summary:")
print(lstm_model)
print(f"\nTotal parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

## 7. Prepare Data for LSTM

Reshape data to (batch_size, sequence_length, features). For simplicity, we'll use sequence_length=1 (treating each match independently but allowing the LSTM architecture).

In [ ]:
# Reshape data for LSTM: (batch_size, seq_len, features)
# Using seq_len=1 for now (can be extended to multi-match sequences later)
X_train_lstm = X_train_scaled.reshape(-1, 1, input_size)
X_val_lstm = X_val_scaled.reshape(-1, 1, input_size)
X_test_lstm = X_test_scaled.reshape(-1, 1, input_size)

print(f"📊 Data reshaped for LSTM:")
print(f"  Train: {X_train_lstm.shape}")
print(f"  Val: {X_val_lstm.shape}")
print(f"  Test: {X_test_lstm.shape}")

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_lstm).to(device)
y_train_tensor = torch.FloatTensor(y_train).to(device)

X_val_tensor = torch.FloatTensor(X_val_lstm).to(device)
y_val_tensor = torch.FloatTensor(y_val).to(device)

X_test_tensor = torch.FloatTensor(X_test_lstm).to(device)
y_test_tensor = torch.FloatTensor(y_test).to(device)

# Create data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"✅ DataLoaders created!")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 8. Train LSTM Model

Train the LSTM with Binary Cross-Entropy loss and Adam optimizer, monitoring validation performance.

In [ ]:
print("🚀 Training LSTM Model...")

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=learning_rate)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_auc': [],
    'val_auc': []
}

# Training loop
best_val_auc = 0
patience = 10
patience_counter = 0

for epoch in range(num_epochs):
    # Training phase
    lstm_model.train()
    train_loss = 0
    train_preds = []
    train_targets = []
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = lstm_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_preds.extend(outputs.detach().cpu().numpy())
        train_targets.extend(batch_y.cpu().numpy())
    
    train_loss /= len(train_loader)
    train_auc = roc_auc_score(train_targets, train_preds)
    
    # Validation phase
    lstm_model.eval()
    val_loss = 0
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = lstm_model(batch_X)
            loss = criterion(outputs, batch_y)
            
            val_loss += loss.item()
            val_preds.extend(outputs.cpu().numpy())
            val_targets.extend(batch_y.cpu().numpy())
    
    val_loss /= len(val_loader)
    val_auc = roc_auc_score(val_targets, val_preds)
    
    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_auc'].append(train_auc)
    history['val_auc'].append(val_auc)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] - "
              f"Train Loss: {train_loss:.4f}, Train AUC: {train_auc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
    
    # Early stopping
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        # Save best model
        best_model_state = lstm_model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⏹️ Early stopping at epoch {epoch+1}")
            break

# Restore best model
lstm_model.load_state_dict(best_model_state)

print(f"\n✅ LSTM Training Complete!")
print(f"Best validation AUC: {best_val_auc:.4f}")